# Approximation Theory
## Stirling, Laplace, and Asymptotic Methods

### Motivation

Many quantities in mathematics and physics — factorials, partition functions, special functions — have no closed-form expression amenable to direct computation. **Approximation theory** provides systematic tools for replacing these exact expressions with computable ones, together with rigorous error bounds.

This notebook develops three deeply connected ideas:

1. **Stirling's approximation** — arguably the most famous asymptotic formula in analysis, giving $n! \approx \sqrt{2\pi n}\,(n/e)^n$
2. **Laplace's method** — a general technique for approximating integrals of the form $\int e^{M f(x)}\,dx$ as $M \to \infty$
3. **Asymptotic series** — divergent expansions that are nevertheless optimally useful when truncated at the right term

All implementations are from scratch using NumPy. `scipy.special.gamma` is used only as ground truth.

### Prerequisites
- Single-variable calculus (Taylor series, integration by parts)
- Complex analysis (steepest descent, saddle points)
- Basic probability (Gaussian integrals)

### References
1. Bender & Orszag — *Advanced Mathematical Methods for Scientists and Engineers* (1978), Chapters 6–7
2. de Bruijn — *Asymptotic Methods in Analysis* (1961)
3. Abramowitz & Stegun — *Handbook of Mathematical Functions* (1964), Chapter 6
4. Olver — *Asymptotics and Special Functions* (1997)
5. Paris & Kaminski — *Asymptotics and Mellin-Barnes Integrals* (2001)

In [ ]:
%matplotlib inline
import numpy as np
import scipy.special as sp
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

SEED = 42
rng  = np.random.default_rng(SEED)

# Colour palette
C_EXACT    = '#2c3e50'   # dark blue-grey  — exact / ground truth
C_STIRLING = '#e74c3c'   # red             — Stirling basic
C_STIRLING2= '#e67e22'   # orange          — Stirling + 1 correction
C_STIRLING3= '#f39c12'   # gold            — Stirling + 2 corrections
C_RAMANUJAN= '#2980b9'   # blue            — Ramanujan
C_LAPLACE  = '#27ae60'   # green           — Laplace method
C_SADDLE   = '#8e44ad'   # purple          — saddle-point

def check(label, condition, fmt=''):
    status = 'PASS' if condition else 'FAIL'
    print(f'  [{status}]  {label}' + (f' ({fmt})' if fmt else ''))

print('Imports OK')

---
## 1.  Problem Statement — Why Approximate?

### 1.1  The Need for Asymptotics

For large $n$, $n!$ overflows 64-bit floats for $n \gtrsim 171$. Combinatorial expressions such as $\binom{2n}{n} \approx 4^n / \sqrt{\pi n}$ arise in probability theory, statistical mechanics, and algorithm analysis.  Direct computation via recursion is $O(n)$ in time and $O(\log n)$ in bits — often impractical.  An explicit formula that is accurate to, say, one part in $10^6$ suffices for most applications.

More broadly, many integrals and sums have the form
$$I(M) = \int_a^b e^{M f(x)}\, g(x)\, dx \qquad M \to \infty$$
where a brute-force numerical integral fails as the integrand becomes sharply peaked.

### 1.2  Asymptotic Notation

| Symbol | Meaning |
|--------|---------|
| $f \sim g$ | $f/g \to 1$ |
| $f = O(g)$ | $\limsup |f/g| < \infty$ |
| $f = o(g)$ | $f/g \to 0$ |

An **asymptotic series** $\sum_{k=0}^\infty a_k / n^k$ is written $f(n) \sim \sum a_k n^{-k}$ if for every fixed $N$:
$$f(n) - \sum_{k=0}^{N} \frac{a_k}{n^k} = O(n^{-(N+1)}) \qquad n \to \infty$$
The series need **not** converge, yet it approximates $f$ to arbitrary relative accuracy for sufficiently large $n$ (with optimal truncation).

---
## 2.  Stirling's Approximation

### 2.1  Statement

$$\boxed{n! \;\sim\; \sqrt{2\pi n}\,\left(\frac{n}{e}\right)^n \qquad n \to \infty}$$

More precisely, the **Stirling series** gives the complete asymptotic expansion:
$$\ln(n!) = n\ln n - n + \tfrac{1}{2}\ln(2\pi n)
  + \frac{1}{12n} - \frac{1}{360n^3} + \frac{1}{1260n^5} - \cdots$$

The coefficients are related to Bernoulli numbers: the $k$-th term is $B_{2k} / [2k(2k-1) n^{2k-1}]$.

### 2.2  Derivation via the Gamma Function

The Gamma function extends the factorial:
$$\Gamma(n+1) = n! = \int_0^\infty t^n e^{-t}\, dt$$

**Step 1 — Substitute $t = n s$:**
$$n! = n^{n+1} \int_0^\infty s^n e^{-ns}\, ds = n^{n+1} \int_0^\infty e^{n(\ln s - s)}\, ds$$

**Step 2 — Identify the saddle point.** Let $f(s) = \ln s - s$. Then $f'(s) = 1/s - 1 = 0$ at $s^* = 1$, with $f(s^*) = -1$ and $f''(s^*) = -1/s^{*2} = -1$.

**Step 3 — Expand around the peak.** For $s = 1 + u/\sqrt{n}$:
$$n f(s) = n f(1) + \tfrac{1}{2} n f''(1) u^2/n + O(u^3/n^{1/2})
          = -n - \tfrac{1}{2}u^2 + O(n^{-1/2})$$

**Step 4 — Gaussian integral.** The dominant contribution gives:
$$n! \approx n^{n+1} e^{-n} \cdot \frac{1}{\sqrt{n}} \int_{-\infty}^\infty e^{-u^2/2}\, du
          = n^{n+1} e^{-n} \cdot \frac{\sqrt{2\pi}}{\sqrt{n}} = \sqrt{2\pi n}\,\left(\frac{n}{e}\right)^n$$

This is exactly Laplace's method — derived in full generality in Section 3.

In [ ]:
# ---------------------------------------------------------------------------
# Stirling's approximation — multiple correction levels
# ---------------------------------------------------------------------------

def log_factorial_exact(n):
    """Log factorial using scipy Gamma as ground truth."""
    return sp.gammaln(np.asarray(n, dtype=float) + 1)

def log_stirling(n, corrections=0):
    """
    Stirling series for log(n!).

    Parameters
    ----------
    n : array-like
    corrections : int
        0 — leading term only: n ln n - n + 0.5 ln(2 pi n)
        1 — add  +1/(12n)
        2 — add  -1/(360 n^3)
        3 — add  +1/(1260 n^5)
    """
    n = np.asarray(n, dtype=float)
    log_n_fact = n * np.log(n) - n + 0.5 * np.log(2 * np.pi * n)
    if corrections >= 1:
        log_n_fact += 1.0 / (12.0 * n)
    if corrections >= 2:
        log_n_fact -= 1.0 / (360.0 * n**3)
    if corrections >= 3:
        log_n_fact += 1.0 / (1260.0 * n**5)
    return log_n_fact

def stirling(n, corrections=0):
    """n! approximation via Stirling's formula."""
    return np.exp(log_stirling(n, corrections))

# Quick verification
test_ns = np.array([5, 10, 20, 50, 100])
print('Stirling relative errors:')
print(f'  {"n":>5}  {"err_0":>10}  {"err_1":>10}  {"err_2":>10}  {"err_3":>10}')
for n in test_ns:
    exact = log_factorial_exact(n)
    errs = [abs(log_stirling(n, c) - exact) / abs(exact) for c in range(4)]
    print(f'  {n:>5}  {errs[0]:10.2e}  {errs[1]:10.2e}  {errs[2]:10.2e}  {errs[3]:10.2e}')

check('Stirling(0) relative error < 1% for n=10',
      abs(log_stirling(10,0) - log_factorial_exact(10)) / log_factorial_exact(10) < 0.01)
check('Stirling(1) relative error < 1e-4 for n=10',
      abs(log_stirling(10,1) - log_factorial_exact(10)) / log_factorial_exact(10) < 1e-4)
check('Stirling(2) relative error < 1e-7 for n=10',
      abs(log_stirling(10,2) - log_factorial_exact(10)) / log_factorial_exact(10) < 1e-7)

---
## 3.  Laplace's Method

### 3.1  General Statement

Let $f : [a,b] \to \mathbb{R}$ have a unique global maximum at an interior point $x^* \in (a,b)$ with $f''(x^*) < 0$. Then as $M \to \infty$:

$$\boxed{\int_a^b e^{M f(x)}\, g(x)\, dx \;\sim\; g(x^*)\, e^{M f(x^*)} \sqrt{\frac{2\pi}{M |f''(x^*)|}} \qquad M \to \infty}$$

**Proof sketch.** Near $x^*$, expand $f(x) = f(x^*) + \frac{1}{2}f''(x^*)(x - x^*)^2 + O((x-x^*)^3)$. The contribution of the tails is exponentially small compared to the peak. Setting $u = \sqrt{M|f''(x^*)|}\,(x - x^*)$:
$$\int_{-\infty}^\infty e^{M f(x^*) - \frac{1}{2}M|f''(x^*)|(x-x^*)^2}\, g(x^*)\, dx = g(x^*) e^{M f(x^*)} \int_{-\infty}^\infty e^{-u^2/2}\, \frac{du}{\sqrt{M|f''(x^*)|}}$$
Using $\int_{-\infty}^\infty e^{-u^2/2} du = \sqrt{2\pi}$ gives the result.

### 3.2  Higher-Order Corrections

Expanding $g(x)$ and $f(x)$ further around $x^*$ yields a systematic series in powers of $1/M$.  Let $f(x) = f(x^*) + \sum_{k\geq 2} \frac{f^{(k)}(x^*)}{k!}(x-x^*)^k$.  The correction at order $1/M$ is:
$$I(M) = g(x^*) e^{M f(x^*)} \sqrt{\frac{2\pi}{M|f''(x^*)|}} \left[1 + \frac{1}{M}\left(\frac{f^{(4)}(x^*)}{8 [f''(x^*)]^2} - \frac{5[f^{(3)}(x^*)]^2}{24[f''(x^*)]^3}\right) + O(M^{-2})\right]$$

### 3.3  Application to $\Gamma(n+1)$

With $f(s) = \ln s - s$, $g(s) = 1$, $s^* = 1$:
- $f(s^*) = -1$,  $f''(s^*) = -1$,  $|f''(s^*)| = 1$
- $f^{(3)}(s^*) = 2$,  $f^{(4)}(s^*) = -6$

Leading term: $n^{n+1} e^{-n} \sqrt{2\pi/n} = \sqrt{2\pi n}\,(n/e)^n$ ✓

First correction (order $1/n$): $1 + \frac{1}{n}\left(\frac{-6}{8} - \frac{5 \cdot 4}{24 \cdot (-1)^3}\right) = 1 + \frac{1}{12n}$ — recovering the first Stirling correction.

In [ ]:
# ---------------------------------------------------------------------------
# Laplace's method — generic numerical implementation
# ---------------------------------------------------------------------------

def laplace_method(f, g, a, b, M_values, n_quad=5000):
    """
    Approximate I(M) = integral_a^b exp(M*f(x)) * g(x) dx
    using Laplace's method (analytic formula).

    Also computes the "exact" value via Gaussian quadrature for comparison.

    Returns
    -------
    approx : array, shape (len(M_values),)  — Laplace approximation
    exact  : array                           — numerical quadrature
    """
    # Locate maximum of f on a fine grid
    x_grid = np.linspace(a, b, 10000)
    x_star = x_grid[np.argmax(f(x_grid))]
    f_star = f(x_star)

    # Numerical second derivative at x_star
    h = 1e-5
    f_pp = (f(x_star + h) - 2*f(x_star) + f(x_star - h)) / h**2
    g_star = g(x_star)

    approx = np.array([
        g_star * np.exp(M * f_star) * np.sqrt(2 * np.pi / (M * abs(f_pp)))
        for M in M_values
    ])

    # Exact via quadrature — work in log space to avoid overflow
    x_q = np.linspace(a, b, n_quad)
    exact = np.array([
        np.trapz(np.exp(M * (f(x_q) - f_star)) * g(x_q), x_q) * np.exp(M * f_star)
        for M in M_values
    ])

    return approx, exact, x_star, f_pp


# Demo: I(M) = integral_0^1 exp(-M x^2) dx
# Exact: sqrt(pi/M)/2 * erf(sqrt(M))
f_demo = lambda x: -x**2
g_demo = lambda x: np.ones_like(x)
M_vals = np.array([1, 2, 5, 10, 20, 50, 100])
approx_demo, exact_demo, x_star_demo, f_pp_demo = laplace_method(
    f_demo, g_demo, 0, 3, M_vals)
exact_analytic = np.sqrt(np.pi / M_vals) / 2 * sp.erf(np.sqrt(M_vals))

print('Laplace method demo: I(M) = int_0^3 exp(-M x^2) dx')
print(f'  Saddle point x* = {x_star_demo:.4f},  f\'\'(x*) = {f_pp_demo:.4f}')
print(f'  {"M":>5}  {"Exact":>14}  {"Laplace":>14}  {"Rel err":>10}')
for M, ex, lap in zip(M_vals, exact_analytic, approx_demo):
    rel_err = abs(lap - ex) / abs(ex)
    print(f'  {M:>5}  {ex:14.6e}  {lap:14.6e}  {rel_err:10.2e}')

check('Laplace relative error < 5% for M=5',
      abs(approx_demo[2] - exact_analytic[2]) / abs(exact_analytic[2]) < 0.05)
check('Laplace relative error < 0.5% for M=50',
      abs(approx_demo[5] - exact_analytic[5]) / abs(exact_analytic[5]) < 0.005)

In [ ]:
# Plot: integrand exp(M f(x)) for various M, illustrating Gaussian peak
M_plot = [1, 5, 20, 100]
x_vis  = np.linspace(0, 3, 500)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
for M in M_plot:
    y = np.exp(M * f_demo(x_vis))
    y /= y.max()
    ax.plot(x_vis, y, label=f'M={M}')
ax.set_xlabel('x')
ax.set_ylabel('Normalised integrand')
ax.set_title("Laplace's Method — Peak Concentration")
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
M_fine = np.logspace(0, 2, 50)
approx_fine, exact_fine, _, _ = laplace_method(f_demo, g_demo, 0, 3, M_fine)
exact_an_fine = np.sqrt(np.pi / M_fine) / 2 * sp.erf(np.sqrt(M_fine))
rel_err_fine = np.abs(approx_fine - exact_an_fine) / np.abs(exact_an_fine)
ax.loglog(M_fine, rel_err_fine, color=C_LAPLACE)
# Reference slope -1 (Laplace error ~ 1/M)
ax.loglog(M_fine, 0.15 / M_fine, 'k--', alpha=0.5, label=r'$O(1/M)$')
ax.set_xlabel('M')
ax.set_ylabel('Relative error')
ax.set_title("Laplace Method — Error Decay $O(1/M)$")
ax.legend()
ax.grid(True, which='both', alpha=0.3)

plt.suptitle("Laplace's Method", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 4.  Ramanujan's Approximation

### 4.1  Formula

Ramanujan's approximation to $n!$ is:

$$\boxed{n! \approx \sqrt{\pi}\,\left(\frac{n}{e}\right)^n \left(8n^3 + 4n^2 + n + \frac{1}{30}\right)^{1/6}}$$

It was recorded in Ramanujan's lost notebook and rediscovered by Berndt. Expanding in powers of $1/n$:
$$\ln(n!) = n\ln n - n + \tfrac{1}{2}\ln(2\pi n)
  + \frac{1}{12n} - \frac{1}{360n^3} + \frac{1}{1260n^5}
  - \frac{1}{1680n^7} + \cdots$$
Ramanujan's formula matches this series through the $1/n^5$ term, beating ordinary Stirling at each level with a more compact expression.

### 4.2  Error Analysis

The relative error of Ramanujan's formula scales as $O(n^{-7})$ versus $O(n^{-1})$ for Stirling's leading term. For $n = 10$, the relative error is roughly $10^{-10}$ — far beyond double-precision floating-point needs.

In [ ]:
# ---------------------------------------------------------------------------
# Ramanujan's approximation
# ---------------------------------------------------------------------------

def log_ramanujan(n):
    """
    Ramanujan's approximation to log(n!).

    log(n!) ~ n log(n) - n + log(pi)/2
              + (1/6) * log(8n^3 + 4n^2 + n + 1/30)
    """
    n = np.asarray(n, dtype=float)
    inner = 8*n**3 + 4*n**2 + n + 1.0/30.0
    return n * np.log(n) - n + 0.5 * np.log(np.pi) + np.log(inner) / 6.0

def ramanujan(n):
    """n! via Ramanujan's formula."""
    return np.exp(log_ramanujan(n))

# Compare all methods
test_ns = np.array([1, 2, 3, 5, 10, 20, 50, 100])

print('Relative errors in log(n!) approximations:')
print(f'  {"n":>4}  {"Stirl-0":>12}  {"Stirl-1":>12}  {"Stirl-2":>12}  {"Ramanujan":>12}')
for n in test_ns:
    ex = log_factorial_exact(n)
    e0 = abs(log_stirling(n, 0) - ex) / abs(ex)
    e1 = abs(log_stirling(n, 1) - ex) / abs(ex)
    e2 = abs(log_stirling(n, 2) - ex) / abs(ex)
    er = abs(log_ramanujan(n)    - ex) / abs(ex)
    print(f'  {n:>4}  {e0:12.3e}  {e1:12.3e}  {e2:12.3e}  {er:12.3e}')

print()
check('Ramanujan better than Stirling-0 for n=5',
      abs(log_ramanujan(5) - log_factorial_exact(5)) <
      abs(log_stirling(5, 0) - log_factorial_exact(5)))
check('Ramanujan better than Stirling-2 for n=5',
      abs(log_ramanujan(5) - log_factorial_exact(5)) <
      abs(log_stirling(5, 2) - log_factorial_exact(5)))
check('Ramanujan relative error < 1e-8 for n=10',
      abs(log_ramanujan(10) - log_factorial_exact(10)) / log_factorial_exact(10) < 1e-8)

---
## 5.  Asymptotic Series

### 5.1  Divergent but Useful

The full Stirling series
$$\ln(n!) \sim n\ln n - n + \tfrac{1}{2}\ln(2\pi n)
  + \sum_{k=1}^\infty \frac{B_{2k}}{2k(2k-1)\, n^{2k-1}}$$
**diverges** for every fixed $n$ (the Bernoulli numbers $B_{2k}$ grow like $(2k)!/(2\pi)^{2k}$, so the terms eventually grow without bound). Yet the series is **asymptotic**: for each fixed truncation $N$, the error is $O(n^{-(2N+1)})$ as $n \to \infty$.

This is the defining property of an **asymptotic expansion**:
$$f(n) \sim \sum_{k=0}^N a_k\, n^{-k} \quad \text{means} \quad
  f(n) - \sum_{k=0}^N a_k n^{-k} = O(n^{-(N+1)}) \text{ as } n\to\infty$$

### 5.2  Bernoulli Numbers and Stirling Coefficients

The Bernoulli numbers $B_0=1, B_2=1/6, B_4=-1/30, B_6=1/42, B_8=-1/30, \ldots$ grow rapidly:
$$|B_{2k}| \sim \frac{2\,(2k)!}{(2\pi)^{2k}} \quad k \to \infty$$
The $k$-th Stirling coefficient $c_k = B_{2k}/[2k(2k-1)]$ satisfies:
$$|c_k| \sim \frac{(2k-2)!}{(2\pi)^{2k-1}}$$

### 5.3  Optimal Truncation

For a fixed $n$, the terms $|c_k|/n^{2k-1}$ initially decrease, reach a minimum at $k^* \approx \pi n / e$, then increase. The minimum term gives the best achievable accuracy:
$$\text{error}_{\min} \approx \exp(-2\pi n / e)$$
Beyond $k^*$, further terms **worsen** the approximation. This is the signature of a **superasymptotic** approximation.

In [ ]:
# ---------------------------------------------------------------------------
# Bernoulli numbers and Stirling series coefficients
# ---------------------------------------------------------------------------

def bernoulli_numbers(K):
    """
    Compute B_0, B_2, B_4, ..., B_{2K} via the recurrence.

    Uses the von Staudt-Clausen / direct sum formula:
    B_m = 1 - sum_{k=0}^{m-1} C(m,k) B_k / (m - k + 1)
    We only need even-indexed values.
    """
    from fractions import Fraction
    B = [Fraction(0)] * (2*K + 2)
    B[0] = Fraction(1)
    for m in range(1, 2*K + 2):
        s = Fraction(0)
        for k in range(m):
            from math import comb
            s += Fraction(comb(m + 1, k)) * B[k]
        B[m] = -s / Fraction(m + 1)
    # Return B_0, B_2, B_4, ..., B_{2K} as floats
    return np.array([float(B[2*k]) for k in range(K+1)])

K_max = 10
B_vals = bernoulli_numbers(K_max)
ks = np.arange(1, K_max + 1)

# Stirling series coefficient: c_k = B_{2k} / (2k * (2k-1))
stirling_coeffs = np.array([
    B_vals[k] / (2*k * (2*k - 1)) for k in ks
])

print('Bernoulli numbers and Stirling coefficients:')
print(f'  {"k":>3}  {"B_{2k}":>20}  {"c_k = B_{2k}/(2k(2k-1))":>24}')
for k, b, c in zip(ks, B_vals[1:K_max+1], stirling_coeffs):
    print(f'  {k:>3}  {b:>20.6e}  {c:>24.6e}')

In [ ]:
# ---------------------------------------------------------------------------
# Optimal truncation demonstration
# ---------------------------------------------------------------------------

def stirling_partial_sum(n, K_max_terms):
    """
    Compute partial sums of Stirling series for log(n!):
      S_K = n ln n - n + 0.5 ln(2 pi n) + sum_{k=1}^K c_k / n^{2k-1}
    
    Returns array of length K_max_terms+1 with S_0, S_1, ..., S_{K_max_terms}.
    """
    n = float(n)
    base = n * np.log(n) - n + 0.5 * np.log(2 * np.pi * n)
    partials = [base]
    correction = 0.0
    for k in range(1, K_max_terms + 1):
        correction += stirling_coeffs[k-1] / n**(2*k - 1)
        partials.append(base + correction)
    return np.array(partials)

exact_log = log_factorial_exact

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: error vs number of terms, for several n
ax = axes[0]
K_terms = np.arange(0, K_max)
ns_demo = [3, 5, 10, 20]
colors_demo = [C_STIRLING, C_STIRLING2, C_LAPLACE, C_RAMANUJAN]
for n_d, col in zip(ns_demo, colors_demo):
    partials = stirling_partial_sum(n_d, K_max - 1)
    errors = np.abs(partials - exact_log(n_d))
    ax.semilogy(K_terms, errors, '-o', color=col, label=f'n={n_d}', markersize=4)
ax.set_xlabel('Number of correction terms K')
ax.set_ylabel('|Error in log(n!)|')
ax.set_title('Stirling Series — Divergence after Optimal Truncation')
ax.legend()
ax.grid(True, alpha=0.3)

# Right: term magnitudes for n=5
ax = axes[1]
n_show = 5
term_sizes = np.abs(stirling_coeffs[:K_max-1]) / n_show**(2*np.arange(1, K_max) - 1)
ax.semilogy(np.arange(1, K_max), term_sizes, '-s', color=C_STIRLING, markersize=6)
# Mark the minimum
k_opt = np.argmin(term_sizes) + 1
ax.axvline(k_opt, color='k', linestyle='--', alpha=0.5, label=f'Optimal k*={k_opt}')
ax.set_xlabel('Term index k')
ax.set_ylabel('|c_k / n^{2k-1}|')
ax.set_title(f'Stirling Term Magnitudes for n={n_show}')
ax.legend()
ax.grid(True, alpha=0.3)

plt.suptitle('Asymptotic Series — Optimal Truncation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Optimal truncation for n=5: k* = {k_opt}')
best_error = np.abs(stirling_partial_sum(5, K_max-1) - exact_log(5)).min()
print(f'Best achievable |error| for n=5: {best_error:.3e}')

check('Stirling series diverges: 8th-term error > 5th-term error for n=5',
      abs(stirling_partial_sum(5, 8)[-1] - exact_log(5)) >
      abs(stirling_partial_sum(5, 5)[-1] - exact_log(5)))

---
## 6.  Saddle-Point Method and Steepest Descent

### 6.1  Extension to Complex Integrals

For a contour integral in the complex plane,
$$I(M) = \int_\mathcal{C} e^{M h(z)}\, g(z)\, dz \qquad M \to \infty,$$
the **saddle-point method** (or **steepest descent**) proceeds as follows:

1. **Find saddle points** $z^*$ where $h'(z^*) = 0$.
2. **Deform the contour** $\mathcal{C}$ to pass through $z^*$ along the **path of steepest descent** — the curve where $\operatorname{Im}[h(z)]$ is constant (the phase is stationary) while $\operatorname{Re}[h(z)]$ decreases most rapidly away from $z^*$.
3. **Local Gaussian integral.** Along the steepest descent path, write $z = z^* + t e^{i\theta}$ where $\theta$ is chosen so that $h''(z^*) e^{2i\theta} < 0$ (real and negative). Then:

$$\boxed{I(M) \sim g(z^*)\, e^{M h(z^*)} \sqrt{\frac{2\pi}{M |h''(z^*)|}}\, e^{i\alpha}
  \qquad \text{where } \alpha = \arg\!\left(e^{i\theta}\right)}$$

### 6.2  Application to the Airy Function

The Airy function $\operatorname{Ai}(x)$ is defined by:
$$\operatorname{Ai}(x) = \frac{1}{2\pi i} \int_{-i\infty}^{i\infty} \exp\!\left(\frac{t^3}{3} - xt\right)\, dt$$

For **large positive $x$**, saddle points of $h(t) = t^3/3 - xt$ satisfy $h'(t) = t^2 - x = 0$, giving $t^* = \pm\sqrt{x}$.

Taking $M = x^{3/2}$ and expanding around $t^* = \sqrt{x}$, steepest descent yields:
$$\operatorname{Ai}(x) \sim \frac{1}{2\sqrt{\pi}\, x^{1/4}} \exp\!\left(-\frac{2}{3} x^{3/2}\right)
  \left[1 - \frac{5}{48 x^{3/2}} + O(x^{-3})\right] \qquad x \to +\infty$$

For **large negative $x$**, both saddle points contribute, giving an oscillatory approximation:
$$\operatorname{Ai}(-x) \sim \frac{1}{\sqrt{\pi}\, x^{1/4}}
  \cos\!\left(\frac{2}{3}x^{3/2} - \frac{\pi}{4}\right) \qquad x \to +\infty$$

In [ ]:
# ---------------------------------------------------------------------------
# Steepest descent path for Gamma integral — visualisation
# ---------------------------------------------------------------------------

def f_gamma(s):
    """f(s) = log(s) - s for the Gamma integrand, broadcast over complex s."""
    return np.log(s) - s

# Real part of f along the steepest descent path through s*=1
# The steepest descent path satisfies Im[f(s)] = Im[f(1)] = -1 (constant)
# Parametrically, near s=1: s = 1 + i*u => Im[log(1+iu) - 1 - iu]
#   = arctan(u) - u ≈ -u^3/3   (small u)
# Exact path found numerically below.

u_vals = np.linspace(-3, 3, 400)
# Path: s = 1 + i*u  (vertical through saddle) — not steepest descent but instructive
s_vertical = 1 + 1j * u_vals
Re_f_vert = np.real(np.log(s_vertical) - s_vertical)

# Steepest descent: Im[f(s)] = Im[f(1)] = -1
# We parametrise: s = x + iy, solve Im[log(s) - s] = -1 numerically
from scipy.optimize import brentq

x_sd = np.linspace(0.05, 4, 400)
y_sd = []
for x in x_sd:
    def eq(y):
        s = complex(x, y)
        return np.imag(np.log(s) - s) + 1  # Im[f(s)] = -1
    try:
        y_sd.append(brentq(eq, 0, 10))
    except Exception:
        y_sd.append(np.nan)
y_sd = np.array(y_sd)

Re_f_sd = np.real(np.log(x_sd + 1j*y_sd) - (x_sd + 1j*y_sd))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: Re[f] along two paths
ax = axes[0]
ax.plot(u_vals, Re_f_vert, color=C_LAPLACE, label='Vertical path s=1+iu')
ax.plot(u_vals,
        np.real(np.log(1 + u_vals) - (1 + u_vals)),
        color=C_STIRLING, label='Real axis path s=1+u')
ax.axhline(-1, color='k', linestyle='--', alpha=0.4, label='f(s*)=-1')
ax.set_xlabel('Path parameter u')
ax.set_ylabel('Re[f(s)]')
ax.set_title('Re[f] Along Paths Through Saddle s*=1')
ax.legend(fontsize=9)
ax.set_ylim(-4, 0)
ax.grid(True, alpha=0.3)

# Right: level curves of Re[f(s)] with steepest descent path
ax = axes[1]
x2d = np.linspace(0.05, 4, 200)
y2d = np.linspace(-3, 3, 200)
X, Y = np.meshgrid(x2d, y2d)
S = X + 1j*Y
RF = np.real(np.log(S) - S)
contour = ax.contourf(X, Y, RF, levels=30, cmap='Blues_r')
ax.contour(X, Y, RF, levels=[-1], colors='red', linestyles='--', linewidths=1.5)
ax.plot(x_sd, y_sd, color=C_SADDLE, linewidth=2, label='Steepest descent')
ax.plot(x_sd, -y_sd, color=C_SADDLE, linewidth=2)
ax.plot(1, 0, 'r*', markersize=12, label='Saddle s*=1')
plt.colorbar(contour, ax=ax, label='Re[log(s)-s]')
ax.set_xlabel('Re(s)')
ax.set_ylabel('Im(s)')
ax.set_title('Level Curves of Re[f(s)] and Steepest Descent Path')
ax.legend(fontsize=9)

plt.suptitle('Saddle-Point / Steepest Descent Geometry', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Airy function: steepest descent approximation vs scipy
# ---------------------------------------------------------------------------

def airy_saddle_pos(x):
    """
    Steepest descent approximation to Ai(x) for large positive x.

    Ai(x) ~ (1 / (2 sqrt(pi) x^{1/4})) * exp(-2/3 * x^{3/2})
             * (1 - 5/(48 x^{3/2}) + ...)
    """
    x = np.asarray(x, dtype=float)
    leading = np.exp(-2.0/3.0 * x**1.5) / (2 * np.sqrt(np.pi) * x**0.25)
    correction = 1.0 - 5.0 / (48.0 * x**1.5)
    return leading * correction

def airy_saddle_neg(x):
    """
    Steepest descent approximation to Ai(-x) for large positive x.

    Ai(-x) ~ (1 / (sqrt(pi) x^{1/4})) * cos(2/3 * x^{3/2} - pi/4)
    """
    x = np.asarray(x, dtype=float)
    return np.cos(2.0/3.0 * x**1.5 - np.pi/4.0) / (np.sqrt(np.pi) * x**0.25)

x_pos = np.linspace(1, 20, 300)
x_neg = np.linspace(1, 20, 300)

Ai_pos_exact  = sp.airy(x_pos)[0]
Ai_pos_approx = airy_saddle_pos(x_pos)

Ai_neg_exact  = sp.airy(-x_neg)[0]
Ai_neg_approx = airy_saddle_neg(x_neg)

fig, axes = plt.subplots(2, 2, figsize=(13, 8))

# Positive argument
ax = axes[0, 0]
ax.semilogy(x_pos, np.abs(Ai_pos_exact),  color=C_EXACT,   label='Exact Ai(x)')
ax.semilogy(x_pos, np.abs(Ai_pos_approx), color=C_SADDLE, linestyle='--', label='Saddle approx')
ax.set_xlabel('x'); ax.set_ylabel('|Ai(x)|')
ax.set_title('Airy Ai(x) — Positive Argument')
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[0, 1]
rel_err_pos = np.abs(Ai_pos_approx - Ai_pos_exact) / np.abs(Ai_pos_exact)
ax.semilogy(x_pos, rel_err_pos, color=C_SADDLE)
ax.semilogy(x_pos, 5.0 / x_pos**1.5, 'k--', alpha=0.5, label=r'$O(x^{-3/2})$')
ax.set_xlabel('x'); ax.set_ylabel('Relative error')
ax.set_title('Saddle Approximation Error for Ai(x)')
ax.legend(); ax.grid(True, alpha=0.3)

# Negative argument
ax = axes[1, 0]
ax.plot(x_neg, Ai_neg_exact,  color=C_EXACT,   label='Exact Ai(-x)')
ax.plot(x_neg, Ai_neg_approx, color=C_SADDLE, linestyle='--', label='Saddle approx', alpha=0.8)
ax.set_xlabel('x'); ax.set_ylabel('Ai(-x)')
ax.set_title('Airy Ai(-x) — Negative Argument (Oscillatory)')
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1, 1]
rel_err_neg = np.abs(Ai_neg_approx - Ai_neg_exact) / (np.abs(Ai_neg_exact) + 1e-30)
# Only plot where exact is not near zero
mask = np.abs(Ai_neg_exact) > 0.01
ax.semilogy(x_neg[mask], rel_err_neg[mask], color=C_SADDLE)
ax.set_xlabel('x'); ax.set_ylabel('Relative error')
ax.set_title('Saddle Approximation Error for Ai(-x)')
ax.grid(True, alpha=0.3)

plt.suptitle('Airy Function: Steepest Descent Approximation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Verification
check('Saddle approx rel error < 1% for Ai(10)',
      abs(airy_saddle_pos(10) - sp.airy(10)[0]) / abs(sp.airy(10)[0]) < 0.01)
check('Saddle approx rel error < 0.1% for Ai(20)',
      abs(airy_saddle_pos(20) - sp.airy(20)[0]) / abs(sp.airy(20)[0]) < 0.001)

---
## 7.  Numerical Experiments — Comprehensive Error Comparison

We compare all approximations to `scipy.special.gammaln` (ground truth) across a wide range of $n$, using the relative error in $\log(n!)$:
$$\varepsilon_{\text{rel}} = \frac{|\hat{L} - \log(n!)|}{|\log(n!)|}
  \qquad \hat{L} \in \{S_0, S_1, S_2, R\}$$

We also examine convergence rates (slope on a log-log plot) to confirm the theoretical predictions:
- $S_0$ error $\sim C_0 / n$ — slope $-1$
- $S_1$ error $\sim C_1 / n^3$ — slope $-3$  
- $S_2$ error $\sim C_2 / n^5$ — slope $-5$
- $R$ error $\sim C_R / n^7$ — slope $-7$

In [ ]:
# ---------------------------------------------------------------------------
# Relative error comparison across all methods
# ---------------------------------------------------------------------------

n_vals = np.logspace(0, 3, 200)  # 1 to 1000

log_exact = log_factorial_exact(n_vals)

err_s0 = np.abs(log_stirling(n_vals, 0) - log_exact) / np.abs(log_exact)
err_s1 = np.abs(log_stirling(n_vals, 1) - log_exact) / np.abs(log_exact)
err_s2 = np.abs(log_stirling(n_vals, 2) - log_exact) / np.abs(log_exact)
err_r  = np.abs(log_ramanujan(n_vals)    - log_exact) / np.abs(log_exact)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: relative error vs n
ax = axes[0]
ax.loglog(n_vals, err_s0, color=C_STIRLING,  lw=1.8, label='Stirling $S_0$')
ax.loglog(n_vals, err_s1, color=C_STIRLING2, lw=1.8, label='Stirling $S_1$')
ax.loglog(n_vals, err_s2, color=C_STIRLING3, lw=1.8, label='Stirling $S_2$')
ax.loglog(n_vals, err_r,  color=C_RAMANUJAN, lw=1.8, label='Ramanujan')

# Reference slopes
nref = np.array([10, 1000])
ax.loglog(nref, 8e-3 / nref,       'k:', alpha=0.4, lw=1, label='$O(n^{-1})$')
ax.loglog(nref, 2e-4 / nref**3,    'k--', alpha=0.4, lw=1, label='$O(n^{-3})$')
ax.loglog(nref, 1e-5 / nref**5,    'k-.', alpha=0.4, lw=1, label='$O(n^{-5})$')
ax.loglog(nref, 5e-8 / nref**7,    'k-', alpha=0.2, lw=1, label='$O(n^{-7})$')

ax.set_xlabel('n')
ax.set_ylabel('Relative error in log(n!)')
ax.set_title('Factorial Approximation — Relative Error')
ax.legend(ncol=2, fontsize=8)
ax.grid(True, which='both', alpha=0.3)

# Right: absolute errors for n=1..30 (small-n regime)
n_small = np.arange(1, 31)
log_ex_s = log_factorial_exact(n_small)

abs_s0 = np.abs(log_stirling(n_small, 0) - log_ex_s)
abs_s1 = np.abs(log_stirling(n_small, 1) - log_ex_s)
abs_s2 = np.abs(log_stirling(n_small, 2) - log_ex_s)
abs_r  = np.abs(log_ramanujan(n_small)    - log_ex_s)

ax = axes[1]
ax.semilogy(n_small, abs_s0, '-o', color=C_STIRLING,  ms=4, lw=1.5, label='Stirling $S_0$')
ax.semilogy(n_small, abs_s1, '-s', color=C_STIRLING2, ms=4, lw=1.5, label='Stirling $S_1$')
ax.semilogy(n_small, abs_s2, '-^', color=C_STIRLING3, ms=4, lw=1.5, label='Stirling $S_2$')
ax.semilogy(n_small, abs_r,  '-D', color=C_RAMANUJAN, ms=4, lw=1.5, label='Ramanujan')
ax.set_xlabel('n')
ax.set_ylabel('Absolute error in log(n!)')
ax.set_title('Small-n Regime (n = 1 to 30)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.suptitle('Factorial Approximation Error Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Estimate convergence rates empirically
# ---------------------------------------------------------------------------

from numpy.polynomial.polynomial import polyfit as polyfit_np

# Use large n where each method has simple power-law error
n_fit = np.logspace(1.5, 3, 100)  # n = 30..1000
log_ex_fit = log_factorial_exact(n_fit)

errs = {
    'Stirling S0': np.abs(log_stirling(n_fit, 0) - log_ex_fit),
    'Stirling S1': np.abs(log_stirling(n_fit, 1) - log_ex_fit),
    'Stirling S2': np.abs(log_stirling(n_fit, 2) - log_ex_fit),
    'Ramanujan':   np.abs(log_ramanujan(n_fit)    - log_ex_fit),
}

expected_slopes = [-1, -3, -5, -7]
print('Empirical convergence rates (expected slope in parentheses):')
print(f'  {"Method":>14}  {"Empirical slope":>16}  {"Expected":>10}')
for (name, err), exp_s in zip(errs.items(), expected_slopes):
    # Linear fit in log-log
    valid = err > 1e-15  # avoid numerical noise floor
    if valid.sum() > 10:
        log_n = np.log10(n_fit[valid])
        log_e = np.log10(err[valid])
        slope = np.polyfit(log_n, log_e, 1)[0]
        print(f'  {name:>14}  {slope:>16.2f}  {exp_s:>10}')
        check(f'{name}: slope within 10% of expected {exp_s}',
              abs(slope - exp_s) / abs(exp_s) < 0.10,
              f'slope={slope:.2f}')
    else:
        print(f'  {name:>14}  (below noise floor)')

In [ ]:
# ---------------------------------------------------------------------------
# Summary table: approximation quality at key values of n
# ---------------------------------------------------------------------------

ns_summary = [1, 2, 5, 10, 20, 50, 100, 500, 1000]

print('Summary: Relative error in log(n!) for each method')
print('=' * 78)
header = f'{"n":>6}  {"Stirling S0":>13}  {"Stirling S1":>13}  '\
         f'{"Stirling S2":>13}  {"Ramanujan":>13}'
print(header)
print('-' * 78)
for n in ns_summary:
    ex = log_factorial_exact(n)
    e0 = abs(log_stirling(n, 0) - ex) / abs(ex)
    e1 = abs(log_stirling(n, 1) - ex) / abs(ex)
    e2 = abs(log_stirling(n, 2) - ex) / abs(ex)
    er = abs(log_ramanujan(n)    - ex) / abs(ex)
    print(f'{n:>6}  {e0:>13.3e}  {e1:>13.3e}  {e2:>13.3e}  {er:>13.3e}')
print('=' * 78)
print('Ground truth: scipy.special.gammaln')

In [ ]:
# ---------------------------------------------------------------------------
# Visual comparison: n! values (log scale) and deviations
# ---------------------------------------------------------------------------

n_vis = np.arange(1, 51)
log_ex_vis = log_factorial_exact(n_vis)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Panel 1: log(n!) values
ax = axes[0]
ax.plot(n_vis, log_ex_vis, color=C_EXACT, lw=2.5, label='Exact log(n!)')
ax.plot(n_vis, log_stirling(n_vis, 0), color=C_STIRLING, lw=1.5,
        linestyle='--', label='Stirling S0', alpha=0.8)
ax.plot(n_vis, log_ramanujan(n_vis), color=C_RAMANUJAN, lw=1.5,
        linestyle=':', label='Ramanujan', alpha=0.9)
ax.set_xlabel('n')
ax.set_ylabel('log(n!)')
ax.set_title('log(n!) Approximations')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Panel 2: deviation log_approx - log_exact
ax = axes[1]
ax.plot(n_vis, log_stirling(n_vis, 0) - log_ex_vis, color=C_STIRLING,  lw=1.8, label='Stirling S0')
ax.plot(n_vis, log_stirling(n_vis, 1) - log_ex_vis, color=C_STIRLING2, lw=1.8, label='Stirling S1')
ax.plot(n_vis, log_stirling(n_vis, 2) - log_ex_vis, color=C_STIRLING3, lw=1.8, label='Stirling S2')
ax.plot(n_vis, log_ramanujan(n_vis)    - log_ex_vis, color=C_RAMANUJAN, lw=1.8, label='Ramanujan')
ax.axhline(0, color='k', lw=0.5)
ax.set_xlabel('n')
ax.set_ylabel('log(approx) - log(exact)')
ax.set_title('Signed Error in log(n!)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Panel 3: relative error (all methods), log scale
ax = axes[2]
n_log = np.logspace(0, 3, 300)
log_ex_log = log_factorial_exact(n_log)
ax.loglog(n_log, np.abs(log_stirling(n_log, 0) - log_ex_log) / np.abs(log_ex_log),
          color=C_STIRLING,  lw=1.8, label='$S_0$')
ax.loglog(n_log, np.abs(log_stirling(n_log, 1) - log_ex_log) / np.abs(log_ex_log),
          color=C_STIRLING2, lw=1.8, label='$S_1$')
ax.loglog(n_log, np.abs(log_stirling(n_log, 2) - log_ex_log) / np.abs(log_ex_log),
          color=C_STIRLING3, lw=1.8, label='$S_2$')
ax.loglog(n_log, np.abs(log_ramanujan(n_log)   - log_ex_log) / np.abs(log_ex_log),
          color=C_RAMANUJAN, lw=1.8, label='Ramanujan')
ax.set_xlabel('n')
ax.set_ylabel('Relative error')
ax.set_title('Relative Error (log-log scale)')
ax.legend(fontsize=9)
ax.grid(True, which='both', alpha=0.3)

plt.suptitle('Factorial Approximation — Visual Summary', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 8.  Summary and References

### Summary

| Method | Formula | Leading error | Good for |
|--------|---------|--------------|----------|
| Stirling $S_0$ | $\sqrt{2\pi n}(n/e)^n$ | $O(n^{-1})$ | $n \geq 20$ |
| Stirling $S_1$ | $S_0 \cdot e^{1/(12n)}$ | $O(n^{-3})$ | $n \geq 5$ |
| Stirling $S_2$ | $S_1 \cdot e^{-1/(360n^3)}$ | $O(n^{-5})$ | $n \geq 3$ |
| Ramanujan | $\sqrt{\pi}(n/e)^n(8n^3+4n^2+n+1/30)^{1/6}$ | $O(n^{-7})$ | $n \geq 1$ |

**Key concepts:**

- **Laplace's method** approximates $\int e^{Mf(x)}dx$ by a Gaussian centered at the maximum of $f$, with error $O(1/M)$ for the leading term. The Stirling approximation is Laplace's method applied to $\Gamma(n+1)$.

- **Stirling's series** is asymptotic (not convergent) — terms grow like $(2k)!$ for large $k$. Optimal truncation at the smallest term gives the best approximation, with residual error $\approx e^{-2\pi n}$.

- **Ramanujan's formula** is remarkably compact yet matches five terms of the Stirling series, achieving $O(n^{-7})$ relative error with a single algebraic expression.

- The **saddle-point method** extends Laplace's method to complex contour integrals, yielding asymptotic approximations to special functions (Airy, Bessel, etc.) with error $O(x^{-3/2})$.

### References

1. Bender, C. M. & Orszag, S. A. — *Advanced Mathematical Methods for Scientists and Engineers* (1978). McGraw-Hill. Chapters 6–7.
2. de Bruijn, N. G. — *Asymptotic Methods in Analysis* (1961). North-Holland.
3. Abramowitz, M. & Stegun, I. A. — *Handbook of Mathematical Functions* (1964). Dover. Chapter 6 (Gamma function), Chapter 10 (Airy functions).
4. Olver, F. W. J. — *Asymptotics and Special Functions* (1997). AK Peters.
5. Paris, R. B. & Kaminski, D. — *Asymptotics and Mellin-Barnes Integrals* (2001). Cambridge.
6. Berndt, B. C. — *Ramanujan's Notebooks, Part I* (1985). Springer.
7. Boyd, J. P. — *The Devil's Invention: Asymptotic, Superasymptotic and Hyperasymptotic Series* (1999). Acta Applicandae Mathematicae 56, 1–98.